In [1]:
!pip install -U transformers==4.45.2 huggingface_hub==0.25.2
!pip install -U datasets evaluate sentencepiece
!pip install sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 49.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.0 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset, DatasetDict
import evaluate
import torch

2025-10-14 17:40:00.049867: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760463600.241399      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760463600.292991      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
df = pd.read_csv("/kaggle/input/en-vn-dataset/data.csv")
df = df.dropna()
df

,en,vi
0,What is a Fenqing ?,Fenqing là gì ?
1,Fenqing is a Chinese word which literally mean...,"Fenqing là một từ tiếng Hoa mà nghĩa đen là "" ..."
2,This word has many translations in English suc...,Từ này có nhiều cách dịch sang tiếng Anh như l...
3,I personally like to call them mob youth or ig...,Cá nhân tôi thích gọi chúng là bọn thanh niên ...
4,It is impossible to understand China without k...,Không thể hiểu được Trung Quốc nếu không biết ...
...,...,...
90611,The Asha 305 will be available in the second q...,"Asha 305 sẽ được bán ra trong quý 2 , trong kh..."
90612,Nokia has to compete with a growing number of ...,Nokia đang phải cạnh tranh với ngày càng nhiều...
90613,"For example , Vodafone recently launched the S...","Ví dụ như , Vodafone gần đây vừa ra mắt chiếc ..."
90614,It can access the Internet using Wi-Fi and HSP...,Smart II có thể truy cập internet qua Wi - Fi ...


In [4]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
test_df, val_df = train_test_split(test_df, test_size=0.5, random_state=42)

print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(63431, 2)
(13592, 2)
(13593, 2)


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
model_name = "Helsinki-NLP/opus-mt-en-vi"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/289M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [7]:
text = "If this world was mine , I'd take your dreams and make 'em multiply . If this world was mine , I'd take your enemies in front of God"
inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_length=128)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Nếu thế giới này là của ta, ta sẽ lấy những giấc mơ của ngươi và làm cho chúng nhân lên. Nếu thế giới này là của ta, ta sẽ mang kẻ thù của ngươi trước mặt Thiên Chúa.


In [8]:
def preprocess_function(examples):
    # Tokenize en (input)
    model_inputs = tokenizer(
        examples["en"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    # Tokenize vi (output)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["vi"],
            max_length=128,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [9]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/63431 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:4109: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/13593 [00:00<?, ? examples/s]

Map:   0%|          | 0/13592 [00:00<?, ? examples/s]

In [10]:
metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = [[tokenizer.decode(l, skip_special_tokens=True)] for l in labels]
    result = metric.compute(predictions=decoded_preds, references=labels)
    return {"bleu": result["score"]}

In [11]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_total_limit=1,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_steps=50,
    disable_tqdm=False,
    report_to="none",     
    push_to_hub=False, 
)

In [12]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,0.339800,0.297396,38.061817
2,0.250700,0.260911,41.500602
3,0.219800,0.238083,44.468214
4,0.185400,0.225672,46.571435
5,0.162800,0.221402,47.266804


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2618: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[53684]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=19825, training_loss=0.24488394608563763, metrics={'train_runtime': 8295.4304, 'train_samples_per_second': 38.232, 'train_steps_per_second': 2.39, 'total_flos': 1.075103592873984e+16, 'train_loss': 0.24488394608563763, 'epoch': 5.0})

In [14]:
trainer.save_model("./fine_tuned_en_vi")
tokenizer.save_pretrained("./fine_tuned_en_vi")

('./fine_tuned_en_vi/tokenizer_config.json',
 './fine_tuned_en_vi/special_tokens_map.json',
 './fine_tuned_en_vi/vocab.json',
 './fine_tuned_en_vi/source.spm',
 './fine_tuned_en_vi/target.spm',
 './fine_tuned_en_vi/added_tokens.json')

In [20]:
sentences = [
    "She would have gone to the meeting if she had known it was so important.",
    "The rapid advancement of technology poses ethical dilemmas that society has yet to fully address.",
    "Consciousness might not merely emerge from neural activity, but from the intricate patterns of information processing within the brain.",
    "The city slept under a blanket of fog, its heartbeat echoing faintly in the rhythm of distant footsteps.",
    "He claimed to be a man of vision, yet somehow he failed to see the mess right in front of him."
]

inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to("cuda")
outputs = model.generate(**inputs, max_length=128)
translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
for en, vi in zip(sentences, translations):
    print("English:", en)
    print("Vietnamese:", vi)
    print("-" * 80)

English: She would have gone to the meeting if she had known it was so important.
Vietnamese: Cô ấy lẽ ra đã đến cuộc họp nếu biết rằng việc này rất quan trọng .
--------------------------------------------------------------------------------
English: The rapid advancement of technology poses ethical dilemmas that society has yet to fully address.
Vietnamese: Sự tiến bộ nhanh chóng của công nghệ đưa ra những vấn đề đạo đức mà xã hội vẫn chưa giải quyết đầy đủ .
--------------------------------------------------------------------------------
English: Consciousness might not merely emerge from neural activity, but from the intricate patterns of information processing within the brain.
Vietnamese: Sự đa dạng có thể không đơn thuần xuất hiện từ hoạt động thần kinh , mà còn từ các mô hình phức tạp xử lý thông tin trong não .
--------------------------------------------------------------------------------
English: The city slept under a blanket of fog, its heartbeat echoing faintly in the rh

In [29]:
sentences = [
    "Hey, Roman numeral seven, bae, drop it like it's hot",
    "If this world was mine, I'd take your dreams and make 'em multiply",
    "If this world was mine, I'd take your enemies in front of God",
    "Introduce 'em to that light, hit them strictly with that fire",
    "In this world, concrete flowers grow",
    "Heartache, she only doin' what she know",
    "Weekends, get it poppin' on the low",
    "Better days comin' for sure"
]

inputs = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to("cuda")
outputs = model.generate(**inputs, max_length=128)
translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
for en, vi in zip(sentences, translations):
    print("English:", en)
    print("Vietnamese:", vi)
    print("-" * 80)

English: Hey, Roman numeral seven, bae, drop it like it's hot
Vietnamese: Này , Roman numeral 7 , bae , thả nó xuống như nóng bỏng
--------------------------------------------------------------------------------
English: If this world was mine, I'd take your dreams and make 'em multiply
Vietnamese: Nếu thế giới này là của tôi , tôi sẽ lấy những giấc mơ của các em và làm cho chúng sinh sôi nảy nở .
--------------------------------------------------------------------------------
English: If this world was mine, I'd take your enemies in front of God
Vietnamese: Nếu thế giới này là của tôi , tôi sẽ đón nhận kẻ thù của ông trước Chúa trời
--------------------------------------------------------------------------------
English: Introduce 'em to that light, hit them strictly with that fire
Vietnamese: Hãy giới thiệu chúng với ánh sáng đó , tấn công hoàn toàn chúng bằng ngọn lửa đó
--------------------------------------------------------------------------------
English: In this world, concrete

In [30]:
!zip -r fine_tuned_en_vi.zip /kaggle/working/fine_tuned_en_vi

updating: kaggle/working/fine_tuned_en_vi/ (stored 0%)
updating: kaggle/working/fine_tuned_en_vi/target.spm (deflated 50%)
updating: kaggle/working/fine_tuned_en_vi/model.safetensors (deflated 7%)
updating: kaggle/working/fine_tuned_en_vi/vocab.json (deflated 70%)
updating: kaggle/working/fine_tuned_en_vi/special_tokens_map.json (deflated 35%)
updating: kaggle/working/fine_tuned_en_vi/tokenizer_config.json (deflated 68%)
updating: kaggle/working/fine_tuned_en_vi/source.spm (deflated 51%)
updating: kaggle/working/fine_tuned_en_vi/training_args.bin (deflated 51%)
updating: kaggle/working/fine_tuned_en_vi/generation_config.json (deflated 43%)
updating: kaggle/working/fine_tuned_en_vi/config.json (deflated 61%)
